# t0d0

* add one more column with the argmax of the q05
* Come up with a more lenient metric that will give you about 70% annotation

# Packages

In [1]:
import pandas as pd
import numpy as np 
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
import os
import spatialdata as spd

from matplotlib import rcParams

rcParams['pdf.fonttype'] = 42 # enables correct plotting of text for PDFs

/home/janzules/mamba_envs/spatial_gpu_py311/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/janzules/mamba_envs/spatial_gpu_py311/lib/python3.11/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


# Data

# Locations

In [2]:
# setting up addresses

# proj_folder       = Path("/coh_labs/yunroseli/Jona/CAR-T/results/cell2location/FirstPass/") # Cohorts
# proj_folder       = Path("/home/janzules/spatial/CAR-T/data/cell2location") # Gemini - first pass files
# proj_folder     = Path("/Users/janzules/Roselab/Spatial/CAR_T/data/cell2location/")
proj_folder   = Path("/coh_labs/yunroseli/Jona/CAR-T/") # Most up to date files

out_figs          = proj_folder / "figures/manual_qc"
adata_out_folder  = proj_folder / "annotated_adata"
adata_out_folder.mkdir(exist_ok=True)
out_figs.mkdir(exist_ok=True)
# Files
zarr_file         = proj_folder / "data/zarr/CellCharterClusters"
adata_ref        = proj_folder / "results/cell2location/C2L_inputs/predicted_cells_lvl2_epochs_200_Ncells1_decalpha_100.h5ad"
# adata_out         = adata_out_folder / "sp_200_epochs_defaultalpha.h5ad" # moved to the save section under annotation



## Processing data

In [3]:
# Load data
adata_vis = sc.read_h5ad(adata_ref)
zdata = spd.read_zarr(zarr_file)

/home/janzules/mamba_envs/spatial_gpu_py311/lib/python3.11/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at .DS_Store is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


In [4]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

def normalize_celltype_cols(df: pd.DataFrame) -> pd.DataFrame:
    # strips everything up to and including "_sf_" so columns become just: T_cell, NK_cell, ...
    new_cols = [re.sub(r"^.*?_sf_", "", c) for c in df.columns]
    out = df.copy()
    out.columns = new_cols
    return out

means_df = normalize_celltype_cols(adata_vis.obsm["means_cell_abundance_w_sf"])
q05_df   = normalize_celltype_cols(adata_vis.obsm["q05_cell_abundance_w_sf"])
q95_df   = normalize_celltype_cols(adata_vis.obsm["q95_cell_abundance_w_sf"])

# force same set + same order
common = sorted(set(means_df.columns) & set(q05_df.columns) & set(q95_df.columns))
means_df = means_df[common]
q05_df   = q05_df[common]
q95_df   = q95_df[common]

assert (means_df.columns == q05_df.columns).all()
assert (means_df.columns == q95_df.columns).all()


# Annotating

In [5]:
def annotate_cell2location_dom(
    means_df: pd.DataFrame,
    q05_df: pd.DataFrame,
    q95_df: pd.DataFrame,
    q_cut: float = 0.05,
    dom_cut: float = 1.3,
    dom_cons_cut: float = 0.35,
    sep_cut: float = -np.inf,
    min_total_mean: float = 0.25,
    unknown_label: str = "Unknown",
):
    assert (means_df.columns == q05_df.columns).all()
    assert (means_df.columns == q95_df.columns).all()

    M = means_df.to_numpy()
    Q05 = q05_df.to_numpy()
    Q95 = q95_df.to_numpy()
    n, k = M.shape

    # winner by mean
    win_idx = M.argmax(axis=1)
    win_mean = M[np.arange(n), win_idx]
    win_q05  = Q05[np.arange(n), win_idx]

    total_mean = M.sum(axis=1)

    # runner-up by mean
    top2_idx = np.argpartition(M, -2, axis=1)[:, -2:]
    ru_idx = np.where(top2_idx[:, 0] == win_idx, top2_idx[:, 1], top2_idx[:, 0])
    ru_mean = M[np.arange(n), ru_idx]
    ru_q95  = Q95[np.arange(n), ru_idx]

    dom = win_mean / (ru_mean + 1e-12)
    dom_cons = win_q05 / (ru_q95 + 1e-12)
    sep = win_q05 - ru_q95

    ok = (
        (total_mean >= min_total_mean) &
        (win_q05 >= q_cut) &
        (dom >= dom_cut) &
        (dom_cons >= dom_cons_cut) &
        (sep >= sep_cut)
    )

    celltypes = means_df.columns.to_numpy()
    labels = np.where(ok, celltypes[win_idx], unknown_label)

    out = pd.DataFrame({
        "label": labels,
        "winner_celltype": celltypes[win_idx],
        "winner_mean": win_mean,
        "winner_q05": win_q05,
        "runnerup_celltype": celltypes[ru_idx],
        "runnerup_mean": ru_mean,
        "runnerup_q95": ru_q95,
        "dom_top1_over_top2": dom,
        "dom_conservative": dom_cons,
        "sep_q05_minus_runnerup_q95": sep,
        "total_abundance_mean": total_mean,
        "is_high_conf": ok,
    }, index=means_df.index)

    return out


In [13]:
ann_perm = annotate_cell2location_dom(
    means_df,
    q05_df,
    q95_df,
    q_cut=0.01,
    dom_cut=1.2,
    dom_cons_cut=0.35,
    sep_cut=-np.inf,
    min_total_mean=0.2,
)

labeled = (ann_perm["label"] != "Unknown").sum()
total = len(ann_perm)
print(f"Annotated: {labeled:,} / {total:,} ({labeled/total*100:.1f}%)")

labeled = (ann_perm["label"] != "Unknown").sum()
total = len(ann_perm)
print(f"Annotated: {labeled:,} / {total:,} ({labeled/total*100:.1f}%)\n")
print(ann_perm["label"].value_counts().to_string())

# print("Permissive labeled_frac:", (adata_vis.obs["c2l_label_perm"] != "Unknown").mean())

Annotated: 186,513 / 425,859 (43.8%)
Annotated: 186,513 / 425,859 (43.8%)

label
Unknown              239346
Cancer_cell           76636
N2_like_Neu           22672
M2_like_Mac           16406
Erythrocyte           13881
Fibroblast            13700
Classical_Mono        10451
N1_like_Neu           10168
Endothelial            6076
NK                     4505
CD8_T                  3421
M1_like_Mac            3228
B                      2379
cDC                    1664
CD4_T                   380
Intermediate_Mac        296
Nonclassical_Mono       225
pDC                     171
NKT                     165
Treg                     89


In [28]:
# 1) Reindex to match zarr order (safe even if order differs)
z_adata = zdata.tables["segmentation_counts"]  # or whatever your table key is
adata_aligned = adata_vis[z_adata.obs_names]   # reorder rows to match zarr

# 2) Annotation columns → .obs
z_adata.obs["c2l_winner"]      = ann_perm["label"].reindex(z_adata.obs_names).values
z_adata.obs["c2l_runnerup"] = ann_perm["runnerup_celltype"].reindex(z_adata.obs_names).values
z_adata.obs["c2l_is_high_conf"]    = ann_perm["is_high_conf"].reindex(z_adata.obs_names).values

# 3) Full abundance matrices → .obsm (DataFrames preserve column names in zarr)
z_adata.obsm["c2l_means"] = means_df.reindex(z_adata.obs_names)
z_adata.obsm["c2l_q05"]   = q05_df.reindex(z_adata.obs_names)
z_adata.obsm["c2l_q95"]   = q95_df.reindex(z_adata.obs_names)

# 4) Write to a new path first, verify, then swap
# zdata.write_zarr(str(zarr_file) + "_annotated")

In [29]:
import numpy as np
import pandas as pd

# --- 1) Row alignment ---
assert z_adata.n_obs == adata_vis.n_obs, "n_obs mismatch"
assert set(z_adata.obs_names) == set(adata_vis.obs_names), "obs_names sets differ"
print(f"[OK] Row alignment: {z_adata.n_obs:,} cells, same obs_names set")

# --- 2) .obs annotation columns ---
for col, src in [
    ("c2l_winner",        ann_perm["label"]),
    ("c2l_runnerup", ann_perm["runnerup_celltype"]),
    ("c2l_is_high_conf",      ann_perm["is_high_conf"]),
]:
    assert col in z_adata.obs.columns, f"{col} missing from z_adata.obs"
    left  = z_adata.obs[col].astype(str).values
    right = src.reindex(z_adata.obs_names).astype(str).values
    n_mismatch = (left != right).sum()
    n_nan = pd.isna(z_adata.obs[col]).sum()
    assert n_mismatch == 0, f"{col}: {n_mismatch} mismatched values"
    print(f"[OK] {col}: 0 mismatches, {n_nan} NaN")

# --- 3) .obsm abundance matrices ---
for key, src_df in [
    ("c2l_means", means_df),
    ("c2l_q05",   q05_df),
    ("c2l_q95",   q95_df),
]:
    assert key in z_adata.obsm, f"{key} missing from z_adata.obsm"
    dst = z_adata.obsm[key]
    assert dst.shape == src_df.shape, f"{key}: shape {dst.shape} vs {src_df.shape}"

    # Column names preserved (only meaningful if stored as DataFrame)
    if isinstance(dst, pd.DataFrame):
        assert list(dst.columns) == list(src_df.columns), f"{key}: column names differ"
        dst_arr = dst.loc[z_adata.obs_names].to_numpy()
    else:
        dst_arr = np.asarray(dst)

    src_arr = src_df.reindex(z_adata.obs_names).to_numpy()
    max_diff = np.nanmax(np.abs(dst_arr - src_arr))
    n_nan = np.isnan(dst_arr).sum()
    assert max_diff < 1e-10, f"{key}: max abs diff = {max_diff}"
    print(f"[OK] {key}: shape={dst.shape}, max|diff|={max_diff:.2e}, NaN={n_nan}")

# --- 4) Spot-check 5 random cells end-to-end ---
rng = np.random.default_rng(0)
sample = rng.choice(z_adata.obs_names, size=5, replace=False)
print("\nSpot-check (random 5 cells):")
print(z_adata.obs.loc[sample, ["c2l_winner", "c2l_runnerup", "c2l_is_high_conf"]])

# --- 5) Label distribution sanity ---
print("\nLabel distribution (z_adata):")
print(z_adata.obs["c2l_winner"].value_counts().head(10).to_string())
labeled_frac = (z_adata.obs["c2l_winner"] != "Unknown").mean()
print(f"\nLabeled fraction: {labeled_frac*100:.1f}%")


[OK] Row alignment: 425,859 cells, same obs_names set
[OK] c2l_winner: 0 mismatches, 0 NaN
[OK] c2l_runnerup: 0 mismatches, 0 NaN
[OK] c2l_is_high_conf: 0 mismatches, 0 NaN
[OK] c2l_means: shape=(425859, 19), max|diff|=0.00e+00, NaN=0
[OK] c2l_q05: shape=(425859, 19), max|diff|=0.00e+00, NaN=0
[OK] c2l_q95: shape=(425859, 19), max|diff|=0.00e+00, NaN=0

Spot-check (random 5 cells):
                            c2l_winner c2l_runnerup  c2l_is_high_conf
F07840_cellid_000061815-1  Cancer_cell  M2_like_Mac              True
F08542_cellid_000027689-1      Unknown  N2_like_Neu             False
F08542_cellid_000005279-1  M2_like_Mac  N2_like_Neu              True
F08542_cellid_000038074-1  Cancer_cell  Erythrocyte              True
F07840_cellid_000145442-1  N2_like_Neu  Cancer_cell              True

Label distribution (z_adata):
c2l_winner
Unknown           239346
Cancer_cell        76636
N2_like_Neu        22672
M2_like_Mac        16406
Erythrocyte        13881
Fibroblast         13700
Cla

In [33]:
z_adata.obs.drop(columns=["c2l_label_perm", "c2l_label_strict", "c2l_runnerup_celltype"], inplace=True)

print("obs columns with c2l:", [c for c in z_adata.obs.columns if c.startswith("c2l_")])

obs columns with c2l: ['c2l_is_high_conf', 'c2l_winner', 'c2l_runnerup']


## Saving

In [ ]:
# --- Save ---

# 1) Push the updated table back into the SpatialData object
zdata.tables["segmentation_counts"] = z_adata

# 2) Write to a new path first (safe — keeps original intact)
out_zarr = zarr_file.parent / (zarr_file.name + "_c2l_annotated")
zdata.write(str(out_zarr))
print("Wrote to:", out_zarr)

# 3) Quick verify — reload and spot-check
import spatialdata as spd
zdata_check = spd.read_zarr(str(out_zarr))
t = zdata_check.tables["segmentation_counts"]


Wrote to: /coh_labs/yunroseli/Jona/CAR-T/data/zarr/CellCharterClusters_c2l_annotated
Reloaded shape: (425859, 19026)
obs columns with c2l: ['c2l_is_high_conf', 'c2l_winner', 'c2l_runnerup']
obsm keys: ['X_cc_1', 'c2l_means', 'c2l_q95', 'X_cc_2', 'spatial', 'X_cc_012', 'X_cc_01', 'X_scVI', 'c2l_q05']


KeyError: 'c2l_label_perm'

In [36]:

print("Reloaded shape:", t.shape)
print("obs columns with c2l:", [c for c in t.obs.columns if c.startswith("c2l_")])
print("obsm keys:", list(t.obsm.keys()))
print(t.obs["c2l_winner"].value_counts().head(5))

Reloaded shape: (425859, 19026)
obs columns with c2l: ['c2l_is_high_conf', 'c2l_winner', 'c2l_runnerup']
obsm keys: ['X_cc_1', 'c2l_means', 'c2l_q95', 'X_cc_2', 'spatial', 'X_cc_012', 'X_cc_01', 'X_scVI', 'c2l_q05']
c2l_winner
Unknown        239346
Cancer_cell     76636
N2_like_Neu     22672
M2_like_Mac     16406
Erythrocyte     13881
Name: count, dtype: int64


In [34]:
out_zarr

PosixPath('/coh_labs/yunroseli/Jona/CAR-T/data/zarr/CellCharterClusters_c2l_annotated')

In [ ]:
out_dir = adata_out_folder / "epoch200_default_first"
out_dir.mkdir(parents=True, exist_ok=True)

# save per-cell annotation tables (includes scores like dom/sep)
ann_perm.to_csv(out_dir / "cell2location_annotations_permissive.csv")
ann_strict.to_csv(out_dir / "cell2location_annotations_strict.csv")

# OPTIONAL: save a smaller CSV with just labels + treatment
adata_vis.obs[["treatment","c2l_label_perm","c2l_label_strict"]].to_csv(
    out_dir / "cell2location_labels_only.csv"
)

# OPTIONAL: write updated AnnData so labels persist
# (choose your preferred filename)
adata_vis.write(out_dir / "adata_fully_annotated_first.h5ad")
print("Wrote outputs to:", out_dir.resolve())


/home/janzules/mamba_envs/spatial_gpu_py311/lib/python3.11/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


Wrote outputs to: /home/janzules/spatial/CAR-T/data/cell2location/annotated_adata/epoch200_default_first


# transfering labels to original adata file

In [ ]:
import spatialdata as spd

/home/janzules/mamba_envs/spatial_gpu_py311/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/janzules/mamba_envs/spatial_gpu_py311/lib/python3.11/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [ ]:
# zarr_file = "/home/janzules/spatial/CAR-T/data/cell2location/adata/zarrFiles/CART_centroid"
# zdata = spd.read_zarr(zarr_file)

/tmp/ipykernel_1572653/2843293580.py:2: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  zdata = spd.read_zarr(zarr_file)


In [ ]:
zdata.tables

{'segmentation_counts': AnnData object with n_obs × n_vars = 427296 × 19059
    obs: 'sample', 'cell_id', 'region', 'TMA', 'mouse', 'tissue', 'condition', 'tumor_loc', 'replicate_num'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'}

In [ ]:
adata_vis

AnnData object with n_obs × n_vars = 427296 × 12128
    obs: 'sample', 'cell_id', 'region', 'TMA', 'mouse', 'tissue', 'condition', 'tumor_loc', 'replicate_num', 'treatment', 'batch_core', 'mt_counts', '_indices', '_scvi_batch', '_scvi_labels', 'total RNA counts', 'c2l_label_perm', 'c2l_label_strict'
    var: 'SYMBOL'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'mod', 'spatialdata_attrs', 'log1p'
    obsm: 'means_cell_abundance_w_sf', 'q05_cell_abundance_w_sf', 'q50_cell_abundance_w_sf', 'q95_cell_abundance_w_sf', 'spatial', 'stds_cell_abundance_w_sf'

In [ ]:
import numpy as np
import pandas as pd
import spatialdata as spd

zarr_file = "/home/janzules/spatial/CAR-T/data/cell2location/adata/zarrFiles/CART_centroid"
zdata = spd.read_zarr(zarr_file)

# Grab the original AnnData inside the SpatialData object
z_adata = zdata.tables["segmentation_counts"]

print("adata_vis:", adata_vis.shape)
print("z_adata  :", z_adata.shape)

/tmp/ipykernel_1572653/2985750747.py:6: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  zdata = spd.read_zarr(zarr_file)


adata_vis: (427296, 12128)
z_adata  : (427296, 19059)


In [ ]:
# -----------------------------
# 1) Decide what to transfer
# -----------------------------
# Recommended: transfer any new columns you created (anything present in adata_vis but not in z_adata)
new_cols = adata_vis.obs.columns.difference(z_adata.obs.columns).tolist()

# Also force-transfer your main annotation columns (even if they already exist)
force_cols = [c for c in ["c2l_label_perm", "c2l_label_strict"] if c in adata_vis.obs.columns]

# You can add other columns you care about explicitly here:
# e.g., force_cols += ["c2l_label_strict_clean", "purity", "winner_q05", ...]
cols_to_transfer = sorted(set(new_cols + force_cols))

print(f"New cols in adata_vis not in z_adata: {len(new_cols)}")
print(f"Total cols_to_transfer: {len(cols_to_transfer)}")
print(cols_to_transfer[:25], "..." if len(cols_to_transfer) > 25 else "")

New cols in adata_vis not in z_adata: 9
Total cols_to_transfer: 9
['_indices', '_scvi_batch', '_scvi_labels', 'batch_core', 'c2l_label_perm', 'c2l_label_strict', 'mt_counts', 'total RNA counts', 'treatment'] 


In [ ]:
assert adata_vis.n_obs == z_adata.n_obs, "n_obs differs — cannot do 1:1 transfer safely."

In [ ]:
adata_vis.obs_names.equals(z_adata.obs_names)

True

In [ ]:
if len(cols_to_transfer) == 0:
    print("Nothing new to transfer (adata_vis.obs has no extra columns vs z_adata.obs).")
else:
    # -----------------------------
    # 2) Sanity checks on index
    # -----------------------------
    # Quick checks
    assert adata_vis.n_obs == z_adata.n_obs, "n_obs differs — cannot do 1:1 transfer safely."

    same_names = adata_vis.obs_names.equals(z_adata.obs_names)
    # ✔ Every single barcode matches
    # ✔ Order is identical
    # ✔ Row-wise assignment is exact
    # ✔ No reindexing, joins, or heuristics involved
    # ✔ Zero risk of annotation drift
    
    same_set = set(adata_vis.obs_names) == set(z_adata.obs_names)

    print("Same obs_names AND same order:", same_names)
    print("Same obs_names set (order may differ):", same_set)

    if not same_set:
        # Fallback to joining on stable keys if needed
        # (only run if you ever hit this case)
        raise RuntimeError(
            "obs_names sets differ. If you truly expect same cells, join using a key "
            "like ['sample','cell_id'] or another unique identifier."
        )

    # -----------------------------
    # 3) Transfer
    # -----------------------------
    if same_names:
        # Fast path: exact alignment
        z_adata.obs.loc[:, cols_to_transfer] = adata_vis.obs[cols_to_transfer].to_numpy()
    else:
        # Safe path: reindex adata_vis to z_adata order
        aligned = adata_vis.obs.loc[z_adata.obs_names, cols_to_transfer]
        z_adata.obs.loc[:, cols_to_transfer] = aligned.to_numpy()

    # -----------------------------
    # 4) Post-transfer sanity checks
    # -----------------------------
    # Check random rows across transferred columns
    rng = np.random.default_rng(0)
    idx = rng.choice(z_adata.n_obs, size=20, replace=False)
    check_names = z_adata.obs_names[idx]

    left = z_adata.obs.loc[check_names, cols_to_transfer]
    right = adata_vis.obs.loc[check_names, cols_to_transfer]

    # For robust compare across dtypes
    mism = (left.astype(str) != right.astype(str)).to_numpy().sum()
    print(f"Random-sample mismatches across transferred fields: {mism}")

    # Full-column equality rates (string compare avoids categorical quirks)
    eq_rates = {}
    for c in cols_to_transfer:
        eq = (z_adata.obs[c].astype(str).values == adata_vis.obs.loc[z_adata.obs_names, c].astype(str).values).mean()
        eq_rates[c] = eq
    worst = sorted(eq_rates.items(), key=lambda x: x[1])[:10]
    print("Worst 10 equality rates:", worst)

    assert all(v == 1.0 for v in eq_rates.values()), "Some transferred columns do not match perfectly!"

Same obs_names AND same order: True
Same obs_names set (order may differ): True
Random-sample mismatches across transferred fields: 0
Worst 10 equality rates: [('_indices', np.float64(1.0)), ('_scvi_batch', np.float64(1.0)), ('_scvi_labels', np.float64(1.0)), ('batch_core', np.float64(1.0)), ('c2l_label_perm', np.float64(1.0)), ('c2l_label_strict', np.float64(1.0)), ('mt_counts', np.float64(1.0)), ('total RNA counts', np.float64(1.0)), ('treatment', np.float64(1.0))]


In [ ]:
# -----------------------------
# 5) (Optional) transfer .uns keys you added
# -----------------------------
z_adata.uns["cell2location_provenance"] = {
    "model_name": adata_vis.uns["mod"].get("model_name"),
    "date": adata_vis.uns["mod"].get("date"),
    "factor_names": list(adata_vis.uns["mod"].get("factor_names", [])),
    "n_cells": adata_vis.n_obs,
    "gene_universe_size": adata_vis.n_vars,
    "scvi_uuid": adata_vis.uns.get("_scvi_uuid"),
    "scvi_manager_uuid": adata_vis.uns.get("_scvi_manager_uuid"),
    "log1p": adata_vis.uns.get("log1p"),
}

In [ ]:
z_adata.obs = z_adata.obs.rename(columns={"total RNA counts": "total_RNA_counts"})

# Now this will pass validation
zdata.tables["segmentation_counts"] = z_adata

In [ ]:
zdata.tables["segmentation_counts"]

AnnData object with n_obs × n_vars = 427296 × 19059
    obs: 'sample', 'cell_id', 'region', 'TMA', 'mouse', 'tissue', 'condition', 'tumor_loc', 'replicate_num', '_indices', '_scvi_batch', '_scvi_labels', 'batch_core', 'c2l_label_perm', 'c2l_label_strict', 'mt_counts', 'total_RNA_counts', 'treatment'
    uns: 'spatialdata_attrs', 'cell2location_provenance'
    obsm: 'spatial'

## Saving

In [ ]:
# -----------------------------
# 6) Save back to Zarr
# -----------------------------
# Strongly recommended: write to a NEW path first, then swap if you’re happy.
out_zarr = zarr_file + "_with_annotations"

zdata.write(out_zarr, overwrite=True)

print("Wrote updated SpatialData to:", out_zarr)

# If you truly want to overwrite the original Zarr (risky), uncomment:
# spd.write_zarr(zdata, zarr_file, overwrite=True)
# print("Overwrote original Zarr at:", zarr_file)

Wrote updated SpatialData to: /home/janzules/spatial/CAR-T/data/cell2location/adata/zarrFiles/CART_centroid_with_annotations
